# AI Job Hunter — Autonomous Multi-Agent System

## Capstone Project: LLM Engineering (Weeks 1–8)

This project applies **every major skill** from the 8-week LLM engineering course into one cohesive application. The "AI Job Hunter" is an autonomous multi-agent system that:

- **Discovers jobs** from RSS feeds using GPT structured output (Week 1-2, 3-4)
- **Matches them to your resume** using RAG with ChromaDB + SentenceTransformer (Week 5)
- **Estimates salaries** via an ensemble of frontier + DNN models (Week 6-7)
- **Identifies skill gaps** using Claude via LiteLLM (Week 1-2)
- **Generates cover letters** using a frontier model (Week 1-2)
- **Sends push notifications** via Pushover (Week 8)
- **Orchestrates everything** via an LLM-driven tool-use agentic loop (Week 3-4, 8)

### Architecture
```
JobHunterFramework (coordinator + memory)
├── JobPlanningAgent (manual sequential pipeline)
│   ├── JobScannerAgent        → RSS feeds + GPT structured output
│   ├── ResumeAgent            → RAG: ChromaDB + SentenceTransformer
│   ├── MarketAnalysisAgent    → Ensemble (frontier + DNN)
│   ├── SkillGapAgent          → Claude via LiteLLM
│   ├── CoverLetterAgent       → GPT frontier model
│   └── AlertAgent             → Pushover notifications
└── AutonomousHuntingAgent     → LLM-driven tool-use orchestration
```

## Phase 1: Foundation — Imports, Setup & Data Models

In [ ]:
import os
import sys
import re
import json
import time
import logging
import queue
import threading
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List, Dict, Self
from dotenv import load_dotenv

import requests
import feedparser
import numpy as np
import torch
import torch.nn as nn
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field
from openai import OpenAI
from litellm import completion
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.manifold import TSNE
import chromadb
import gradio as gr
import plotly.graph_objects as go

load_dotenv(override=True)

# Module-level singletons — shared across all agents to avoid redundant init
_openai_client = OpenAI()
_sentence_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("All imports successful!")
print("Shared OpenAI client and SentenceTransformer initialized.")

### Base Agent Class & Log Utilities
Reused from week8 — provides colored logging for each agent.

In [ ]:
# === Base Agent Class (from week8/agents/agent.py) ===

class Agent:
    """Abstract superclass for all Agents — provides colored logging."""

    RED = '\033[31m'
    GREEN = '\033[32m'
    YELLOW = '\033[33m'
    BLUE = '\033[34m'
    MAGENTA = '\033[35m'
    CYAN = '\033[36m'
    WHITE = '\033[37m'
    BG_BLACK = '\033[40m'
    RESET = '\033[0m'

    name: str = ""
    color: str = '\033[37m'

    def log(self, message):
        color_code = self.BG_BLACK + self.color
        message = f"[{self.name}] {message}"
        logging.info(color_code + message + self.RESET)


# === Log Utilities (from week8/log_utils.py) ===

BG_BLUE = '\033[44m'

mapper = {
    Agent.BG_BLACK + Agent.RED: "#dd0000",
    Agent.BG_BLACK + Agent.GREEN: "#00dd00",
    Agent.BG_BLACK + Agent.YELLOW: "#dddd00",
    Agent.BG_BLACK + Agent.BLUE: "#0000ee",
    Agent.BG_BLACK + Agent.MAGENTA: "#aa00dd",
    Agent.BG_BLACK + Agent.CYAN: "#00dddd",
    Agent.BG_BLACK + Agent.WHITE: "#87CEEB",
    BG_BLUE + Agent.WHITE: "#ff7800",
}


def reformat(message):
    """Convert ANSI color codes to HTML spans for Gradio display."""
    for key, value in mapper.items():
        message = message.replace(key, f'<span style="color: {value}">')
    message = message.replace(Agent.RESET, '</span>')
    return message


def init_logging():
    root = logging.getLogger()
    root.setLevel(logging.INFO)
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter(
        "[%(asctime)s] [AI Job Hunter] [%(levelname)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S %z",
    )
    handler.setFormatter(formatter)
    root.addHandler(handler)

print("Agent base class and log utilities ready.")

### Pydantic Data Models
All structured data types for the job hunting pipeline — analogous to `deals.py` from week8.

In [ ]:
# === Pydantic Data Models (jobs.py) ===

class Job(BaseModel):
    """A job posting discovered from an RSS feed."""
    title: str = Field(description="Job title")
    company: str = Field(description="Company name")
    location: str = Field(description="Job location (e.g. Remote, New York, etc.)")
    description: str = Field(description="A 2-3 sentence summary of the job role and responsibilities")
    url: str = Field(description="URL of the job posting")
    salary_text: str = Field(default="", description="Any salary info mentioned in the posting")
    tags: List[str] = Field(default_factory=list, description="Relevant skill tags (e.g. Python, AWS)")


class JobSelection(BaseModel):
    """A list of parsed jobs from RSS scraping."""
    jobs: List[Job] = Field(description="The parsed job postings")


class MatchResult(BaseModel):
    """Result of matching a job against a resume via RAG."""
    job: Job
    match_score: float = Field(description="0-100 score of how well the resume matches the job")
    matched_skills: List[str] = Field(default_factory=list, description="Skills that match between resume and job")
    missing_skills: List[str] = Field(default_factory=list, description="Skills the job requires but resume lacks")
    reasoning: str = Field(default="", description="Brief explanation of the match")


class SalaryEstimate(BaseModel):
    """Salary estimation from the ensemble model."""
    job: Job
    estimated_salary: float = Field(description="Estimated annual salary in USD")
    confidence: str = Field(default="medium", description="low/medium/high confidence")
    frontier_estimate: float = Field(default=0.0, description="Frontier model estimate")
    dnn_estimate: float = Field(default=0.0, description="DNN model estimate")


class SkillGapAnalysis(BaseModel):
    """Analysis of missing skills and recommended learning resources."""
    job: Job
    missing_skills: List[str] = Field(default_factory=list)
    learning_resources: List[Dict[str, str]] = Field(
        default_factory=list,
        description="List of dicts with 'skill', 'resource', 'type' keys"
    )
    priority: str = Field(default="medium", description="Priority level for upskilling")
    estimated_weeks: int = Field(default=4, description="Estimated weeks to close gaps")


class CoverLetter(BaseModel):
    """A generated cover letter for a specific job."""
    job: Job
    letter_text: str = Field(description="The full cover letter text")
    key_points: List[str] = Field(default_factory=list, description="Key selling points highlighted")


class HuntResult(BaseModel):
    """Complete result from a full job hunting run."""
    matches: List[MatchResult] = Field(default_factory=list)
    salary_estimates: List[SalaryEstimate] = Field(default_factory=list)
    skill_gaps: List[SkillGapAnalysis] = Field(default_factory=list)
    cover_letters: List[CoverLetter] = Field(default_factory=list)
    best_match: Optional[MatchResult] = None


print("Data models defined.")
print(f"Models: {', '.join([cls.__name__ for cls in [Job, JobSelection, MatchResult, SalaryEstimate, SkillGapAnalysis, CoverLetter, HuntResult]])}")

### Sample Resume
A demo resume for the AI Job Hunter to match against job postings.

In [ ]:
SAMPLE_RESUME = """
# Kelvin Isievwore — Senior Software Engineer

## Summary
Full-stack software engineer with 10 + years of experience building scalable web applications,
REST APIs, and cloud-native microservices. Passionate about AI/ML, LLM engineering, and
building autonomous agent systems. Currently exploring multi-agent architectures, RAG pipelines,
and fine-tuning LLMs.

## Technical Skills
- **Languages:** Python, JavaScript/TypeScript, Go, SQL
- **AI/ML:** PyTorch, TensorFlow, Hugging Face, LangChain, LiteLLM, OpenAI API, Claude API
- **Web:** React, Next.js, Node.js, FastAPI, Django, Flask
- **Data:** PostgreSQL, MongoDB, Redis, ChromaDB, Pinecone, Elasticsearch
- **Cloud:** AWS (Lambda, ECS, S3, SageMaker), GCP, Docker, Kubernetes
- **DevOps:** CI/CD, GitHub Actions, Terraform, Prometheus, Grafana

## Experience

### Senior Software Engineer — Andela (2021–Present)
- Led development of microservices handling 10M+ daily API requests
- Built RAG-powered internal knowledge base reducing support tickets by 40%
- Designed and deployed ML pipelines for talent matching using embeddings
- Mentored 5 junior engineers on Python best practices and system design

### Software Engineer — TechCorp (2019–2021)
- Developed real-time data processing pipelines with Apache Kafka and Python
- Built full-stack web applications with React and Django REST framework
- Implemented CI/CD pipelines reducing deployment time by 60%

### Junior Developer — StartupXYZ (2018–2019)
- Built REST APIs serving 100K+ monthly active users
- Contributed to open-source Python libraries

## Education
- B.Sc. Computer Science — University of Lagos (2018)
- LLM Engineering Certificate — Andela Learning (2025)

## Certifications
- AWS Solutions Architect Associate
- Google Cloud Professional Data Engineer
"""

print(f"Resume loaded: {len(SAMPLE_RESUME)} characters, {len(SAMPLE_RESUME.split())} words")

## Phase 2: Data Ingestion — Job Scanner & Resume Vectorstore

### Build Vectorstore
Chunks the resume and ingests it into ChromaDB with SentenceTransformer embeddings — mirrors `build_vectorstore.py` from week8.

In [ ]:
# === Build Vectorstore — Resume ingestion into ChromaDB ===

RESUME_DB = "resume_vectorstore"


def chunk_resume(resume_text: str, chunk_size: int = 200) -> List[str]:
    """Split resume into overlapping chunks for embedding."""
    sections = re.split(r'\n## ', resume_text)
    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue
        words = section.split()
        for i in range(0, len(words), chunk_size // 2):
            chunk = " ".join(words[i:i + chunk_size])
            if len(chunk) > 20:
                chunks.append(chunk)
    return chunks


def build_resume_vectorstore(resume_text: str, db_path: str = RESUME_DB, force_rebuild: bool = False) -> chromadb.Collection:
    """Ingest resume chunks into ChromaDB with SentenceTransformer embeddings."""
    client = chromadb.PersistentClient(path=db_path)

    # Check if collection already exists with documents — skip rebuild unless forced
    if not force_rebuild:
        try:
            collection = client.get_collection("resume")
            if collection.count() > 0:
                print(f"Vectorstore already exists with {collection.count()} chunks at '{db_path}' (use force_rebuild=True to rebuild)")
                return collection
        except Exception:
            pass  # Collection doesn't exist yet, proceed with build

    chunks = chunk_resume(resume_text)
    print(f"Created {len(chunks)} resume chunks")

    # Delete existing collection if it exists, then recreate
    try:
        client.delete_collection("resume")
    except Exception:
        pass
    collection = client.get_or_create_collection("resume")

    embeddings = _sentence_model.encode(chunks).astype(float).tolist()
    ids = [f"chunk_{i}" for i in range(len(chunks))]
    metadatas = [{"chunk_index": i, "length": len(c)} for i, c in enumerate(chunks)]

    collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=ids,
        metadatas=metadatas,
    )
    print(f"Vectorstore built with {collection.count()} chunks at '{db_path}'")
    return collection


# Build the vectorstore
resume_collection = build_resume_vectorstore(SAMPLE_RESUME)

### JobScannerAgent
Scrapes job postings from RSS feeds and uses GPT structured output to parse them — mirrors `scanner_agent.py` from week8.

In [ ]:
# === JobScannerAgent — RSS feeds + GPT structured output ===

JOB_FEEDS = [
    "https://remoteok.com/remote-dev-jobs.rss",
    "https://weworkremotely.com/categories/remote-back-end-programming-jobs.rss",
    "https://weworkremotely.com/categories/remote-full-stack-programming-jobs.rss",
]


class ScrapedJob:
    """A job posting retrieved from an RSS feed."""

    def __init__(self, entry: Dict[str, str], feed_source: str = ""):
        self.title = entry.get("title", "Unknown")[:150]
        self.summary = self._extract_text(entry.get("summary", ""))[:800]
        self.url = entry.get("link", entry.get("links", [{}])[0].get("href", ""))
        self.source = feed_source

    @staticmethod
    def _extract_text(html_snippet: str) -> str:
        soup = BeautifulSoup(html_snippet, "html.parser")
        text = soup.get_text(strip=True)
        text = re.sub(r"<[^<]+?>", "", text)
        return text.replace("\n", " ").strip()

    def __repr__(self):
        return f"<{self.title}>"

    def describe(self) -> str:
        return f"Title: {self.title}\nDescription: {self.summary}\nURL: {self.url}"

    @classmethod
    def fetch(cls, show_progress: bool = False) -> List["ScrapedJob"]:
        """Retrieve jobs from all configured RSS feeds."""
        jobs = []
        for feed_url in JOB_FEEDS:
            try:
                feed = feedparser.parse(feed_url)
                source = feed_url.split("/")[2]
                for entry in feed.entries[:10]:
                    jobs.append(cls(entry, source))
                    time.sleep(0.02)
            except Exception as e:
                logging.warning(f"Could not fetch {feed_url}: {e}")
        return jobs


class JobScannerAgent(Agent):
    """Scans RSS feeds for job postings and uses GPT to parse them into structured data."""

    MODEL = "gpt-4o-mini"
    name = "Job Scanner Agent"
    color = Agent.CYAN

    SYSTEM_PROMPT = """You are a job posting parser. Given a list of raw job postings from RSS feeds,
extract and structure each job into a clean format. Focus on:
- Clear job title
- Company name (extract from title or description)
- Location (Remote if not specified)
- A concise 2-3 sentence description of the role
- The job URL
- Any salary information mentioned
- Relevant skill tags (programming languages, frameworks, tools)

Only include jobs that are clearly tech/software roles. Respond strictly in JSON."""

    USER_PROMPT_PREFIX = """Parse these job postings into structured data. Extract company names from titles
(they often follow patterns like "Company Name - Job Title" or "Job Title at Company").
For each job, identify relevant technical skill tags.

Job Postings:

"""

    def __init__(self):
        self.log("Job Scanner Agent is initializing")
        self.openai = _openai_client
        self.log("Job Scanner Agent is ready")

    def fetch_jobs(self, memory_urls: List[str]) -> List[ScrapedJob]:
        self.log("Fetching jobs from RSS feeds")
        scraped = ScrapedJob.fetch()
        result = [job for job in scraped if job.url not in memory_urls]
        self.log(f"Found {len(result)} new job postings")
        return result

    def scan(self, memory_urls: Optional[List[str]] = None) -> Optional[JobSelection]:
        """Scan RSS feeds and parse jobs using GPT structured output."""
        if memory_urls is None:
            memory_urls = []
        scraped = self.fetch_jobs(memory_urls)
        if not scraped:
            self.log("No new jobs found")
            return None

        user_prompt = self.USER_PROMPT_PREFIX
        user_prompt += "\n\n".join([job.describe() for job in scraped[:15]])

        self.log(f"Calling OpenAI to parse {len(scraped[:15])} job postings")
        try:
            result = self.openai.beta.chat.completions.parse(
                model=self.MODEL,
                messages=[
                    {"role": "system", "content": self.SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                response_format=JobSelection,
            )
            parsed = result.choices[0].message.parsed
            self.log(f"Parsed {len(parsed.jobs)} job postings from OpenAI")
            return parsed
        except Exception as e:
            self.log(f"Error parsing jobs: {e}")
            return None

    def test_scan(self) -> JobSelection:
        """Return test data for development/testing."""
        return JobSelection(jobs=[
            Job(
                title="Senior Python Engineer",
                company="TechStartup AI",
                location="Remote",
                description="Build and scale backend services using Python, FastAPI, and PostgreSQL. Lead architecture decisions for ML pipeline infrastructure.",
                url="https://example.com/job/1",
                salary_text="$150,000 - $200,000",
                tags=["Python", "FastAPI", "PostgreSQL", "AWS", "ML"],
            ),
            Job(
                title="Full Stack Developer",
                company="CloudScale Inc",
                location="Remote (US)",
                description="Develop React frontends and Node.js backends for a SaaS platform serving 50K+ users. Strong focus on real-time features and API design.",
                url="https://example.com/job/2",
                salary_text="$120,000 - $160,000",
                tags=["React", "Node.js", "TypeScript", "MongoDB", "Docker"],
            ),
            Job(
                title="ML Engineer",
                company="DataDriven Corp",
                location="Remote (Global)",
                description="Design and deploy ML models for NLP tasks. Build RAG pipelines and fine-tune LLMs. Requires strong PyTorch and Hugging Face experience.",
                url="https://example.com/job/3",
                salary_text="$160,000 - $220,000",
                tags=["Python", "PyTorch", "Hugging Face", "LLM", "RAG", "AWS"],
            ),
            Job(
                title="DevOps Engineer",
                company="InfraCloud",
                location="Remote (Europe)",
                description="Manage Kubernetes clusters, CI/CD pipelines, and cloud infrastructure on AWS/GCP. Automate everything with Terraform and GitHub Actions.",
                url="https://example.com/job/4",
                salary_text="$130,000 - $170,000",
                tags=["Kubernetes", "Docker", "Terraform", "AWS", "GCP", "CI/CD"],
            ),
            Job(
                title="AI/LLM Engineer",
                company="AgentAI Labs",
                location="Remote",
                description="Build autonomous multi-agent systems using LLMs. Implement tool-use patterns, RAG pipelines, and agentic workflows with OpenAI and Claude APIs.",
                url="https://example.com/job/5",
                salary_text="$180,000 - $250,000",
                tags=["Python", "LLM", "OpenAI", "Claude", "RAG", "Agents", "LangChain"],
            ),
        ])


# Quick test
scanner = JobScannerAgent()
test_jobs = scanner.test_scan()
print(f"Test scan returned {len(test_jobs.jobs)} jobs:")
for job in test_jobs.jobs:
    print(f"  - {job.title} @ {job.company} ({job.location})")

## Phase 3: Core RAG — ResumeAgent

Uses SentenceTransformer embeddings to find relevant resume chunks in ChromaDB, then GPT to score the match — mirrors `frontier_agent.py` RAG pattern from week8.

In [ ]:
# === ResumeAgent — RAG matching: resume chunks ↔ job descriptions ===

class ResumeAgent(Agent):
    """Uses RAG to match jobs against a resume stored in ChromaDB."""

    name = "Resume Agent"
    color = Agent.BLUE
    MODEL = "gpt-4o-mini"

    def __init__(self, collection: chromadb.Collection):
        self.log("Initializing Resume Agent")
        self.openai = _openai_client
        self.collection = collection
        self.model = _sentence_model
        self.log("Resume Agent is ready")

    def find_relevant_chunks(self, job_description: str, n_results: int = 5) -> List[str]:
        """Find resume chunks most relevant to this job description."""
        self.log("Performing RAG search in ChromaDB for relevant resume chunks")
        vector = self.model.encode([job_description])
        results = self.collection.query(
            query_embeddings=vector.astype(float).tolist(),
            n_results=n_results,
        )
        documents = results["documents"][0]
        self.log(f"Found {len(documents)} relevant resume chunks")
        return documents

    def match_job(self, job: Job) -> MatchResult:
        """Score how well the resume matches this job using RAG + GPT."""
        relevant_chunks = self.find_relevant_chunks(
            f"{job.title} {job.description} {' '.join(job.tags)}"
        )
        context = "\n\n".join([f"Resume Section:\n{chunk}" for chunk in relevant_chunks])

        prompt = f"""Analyze how well this candidate's resume matches the job posting.

Job Title: {job.title}
Company: {job.company}
Description: {job.description}
Required Skills: {', '.join(job.tags)}

Relevant Resume Sections:
{context}

Respond in JSON with these fields:
- match_score: integer 0-100 (how well the candidate fits)
- matched_skills: list of skills that match between resume and job
- missing_skills: list of skills the job requires but the resume lacks
- reasoning: 1-2 sentence explanation of the match quality"""

        self.log(f"Calling GPT to score match for '{job.title}'")
        try:
            result = self.openai.beta.chat.completions.parse(
                model=self.MODEL,
                messages=[
                    {"role": "system", "content": "You are an expert recruiter analyzing resume-job fit. Be accurate and fair in scoring."},
                    {"role": "user", "content": prompt},
                ],
                response_format=MatchResult,
            )
            match = result.choices[0].message.parsed
            # Ensure the job object is preserved (GPT may alter it)
            match.job = job
            self.log(f"Match score for '{job.title}': {match.match_score}/100")
            return match
        except Exception as e:
            self.log(f"Error matching job: {e}")
            return MatchResult(
                job=job,
                match_score=0,
                matched_skills=[],
                missing_skills=job.tags,
                reasoning=f"Error during matching: {e}",
            )


# Test the ResumeAgent
resume_agent = ResumeAgent(resume_collection)
test_match = resume_agent.match_job(test_jobs.jobs[0])
print(f"\nMatch result for '{test_match.job.title}':")
print(f"  Score: {test_match.match_score}/100")
print(f"  Matched: {test_match.matched_skills}")
print(f"  Missing: {test_match.missing_skills}")
print(f"  Reasoning: {test_match.reasoning}")

## Phase 4: Analysis Agents

### SkillGapAgent
Uses Claude via LiteLLM to analyze missing skills and recommend learning resources.

In [ ]:
# === SkillGapAgent — Claude via LiteLLM for gap analysis ===

class SkillGapAgent(Agent):
    """Analyzes skill gaps between a resume and job requirements using Claude via LiteLLM."""

    name = "Skill Gap Agent"
    color = Agent.MAGENTA
    MODEL = "claude-sonnet-4-5"

    def __init__(self):
        self.log("Skill Gap Agent is initializing")
        self.log("Skill Gap Agent is ready (using Claude via LiteLLM)")

    def analyze(self, match_result: MatchResult, resume_text: str) -> SkillGapAnalysis:
        """Analyze skill gaps and recommend learning resources."""
        job = match_result.job
        self.log(f"Analyzing skill gaps for '{job.title}' at {job.company}")

        prompt = f"""Analyze the skill gaps between this candidate and the job requirements.

Job: {job.title} at {job.company}
Job Description: {job.description}
Required Skills: {', '.join(job.tags)}
Matched Skills: {', '.join(match_result.matched_skills)}
Missing Skills from initial scan: {', '.join(match_result.missing_skills)}

Candidate Resume (summary):
{resume_text[:1500]}

Provide a thorough skill gap analysis in JSON format with these fields:
- missing_skills: list of specific skills the candidate needs to develop
- learning_resources: list of objects with "skill", "resource" (specific course/book/project name), and "type" (course/book/project/certification)
- priority: "high", "medium", or "low" based on how critical the gaps are
- estimated_weeks: integer estimate of weeks needed to close the gaps"""

        try:
            response = completion(
                model=self.MODEL,
                messages=[
                    {"role": "system", "content": "You are a career advisor specializing in tech skills. Be specific and actionable with learning recommendations. Respond in valid JSON."},
                    {"role": "user", "content": prompt},
                ],
                response_format={"type": "json_object"},
            )
            content = response.choices[0].message.content
            data = json.loads(content)

            analysis = SkillGapAnalysis(
                job=job,
                missing_skills=data.get("missing_skills", match_result.missing_skills),
                learning_resources=data.get("learning_resources", []),
                priority=data.get("priority", "medium"),
                estimated_weeks=data.get("estimated_weeks", 4),
            )
            self.log(f"Found {len(analysis.missing_skills)} skill gaps, priority: {analysis.priority}")
            return analysis

        except Exception as e:
            self.log(f"Error in skill gap analysis: {e}")
            return SkillGapAnalysis(
                job=job,
                missing_skills=match_result.missing_skills,
                learning_resources=[],
                priority="unknown",
                estimated_weeks=0,
            )


# Test SkillGapAgent
skill_gap_agent = SkillGapAgent()
test_gap = skill_gap_agent.analyze(test_match, SAMPLE_RESUME)
print(f"\nSkill gap analysis for '{test_gap.job.title}':")
print(f"  Missing skills: {test_gap.missing_skills}")
print(f"  Priority: {test_gap.priority}")
print(f"  Estimated weeks: {test_gap.estimated_weeks}")
print(f"  Resources: {len(test_gap.learning_resources)} recommendations")
for r in test_gap.learning_resources[:3]:
    print(f"    - {r.get('skill', '?')}: {r.get('resource', '?')} ({r.get('type', '?')})")

### CoverLetterAgent
Uses GPT frontier model to generate tailored cover letters — demonstrates Week 1-2 prompt engineering skills.

In [ ]:
# === CoverLetterAgent — GPT frontier model for letter generation ===

class CoverLetterAgent(Agent):
    """Generates tailored cover letters using GPT."""

    name = "Cover Letter Agent"
    color = Agent.YELLOW
    MODEL = "gpt-4o-mini"

    def __init__(self):
        self.log("Cover Letter Agent is initializing")
        self.openai = _openai_client
        self.log("Cover Letter Agent is ready")

    def generate(self, match_result: MatchResult, resume_text: str) -> CoverLetter:
        """Generate a tailored cover letter for the matched job."""
        job = match_result.job
        self.log(f"Generating cover letter for '{job.title}' at {job.company}")

        prompt = f"""Write a compelling, professional cover letter for this job application.

Job Title: {job.title}
Company: {job.company}
Location: {job.location}
Job Description: {job.description}
Required Skills: {', '.join(job.tags)}

Candidate's Matching Skills: {', '.join(match_result.matched_skills)}
Match Reasoning: {match_result.reasoning}

Candidate Resume:
{resume_text[:2000]}

Guidelines:
- Address the letter to the hiring manager at {job.company}
- Highlight the candidate's most relevant experience and skills
- Show enthusiasm for the specific role and company
- Keep it to 3-4 paragraphs
- Be professional but personable
- End with a clear call to action

Also identify 3-5 key selling points that make this candidate stand out.

Respond in JSON with:
- letter_text: the full cover letter
- key_points: list of 3-5 key selling points"""

        try:
            response = self.openai.chat.completions.create(
                model=self.MODEL,
                messages=[
                    {"role": "system", "content": "You are an expert career coach who writes compelling, personalized cover letters. Always respond in valid JSON."},
                    {"role": "user", "content": prompt},
                ],
                response_format={"type": "json_object"},
            )
            data = json.loads(response.choices[0].message.content)

            letter = CoverLetter(
                job=job,
                letter_text=data.get("letter_text", ""),
                key_points=data.get("key_points", []),
            )
            self.log(f"Cover letter generated ({len(letter.letter_text)} chars, {len(letter.key_points)} key points)")
            return letter

        except Exception as e:
            self.log(f"Error generating cover letter: {e}")
            return CoverLetter(job=job, letter_text=f"Error: {e}", key_points=[])


# Test CoverLetterAgent
cover_letter_agent = CoverLetterAgent()
test_letter = cover_letter_agent.generate(test_match, SAMPLE_RESUME)
print(f"\nCover letter for '{test_letter.job.title}':")
print(f"  Key points: {test_letter.key_points}")
print(f"  Letter preview: {test_letter.letter_text[:300]}...")

## Phase 5: Salary Ensemble

### SalaryNeuralNetwork
A PyTorch DNN with residual blocks for salary prediction — mirrors `deep_neural_network.py` from week8 (Week 6 skills).

In [ ]:
# === SalaryNeuralNetwork — PyTorch DNN for salary prediction ===

class ResidualBlock(nn.Module):
    """Residual block with skip connections — mirrors week8 architecture."""
    def __init__(self, hidden_size, dropout_prob):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.block(x) + x)


class SalaryDNN(nn.Module):
    """Deep Neural Network for salary prediction from job text features."""
    def __init__(self, input_size=5000, num_layers=6, hidden_size=2048, dropout_prob=0.2):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
        )
        self.residual_blocks = nn.ModuleList([
            ResidualBlock(hidden_size, dropout_prob) for _ in range(num_layers - 2)
        ])
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.input_layer(x)
        for block in self.residual_blocks:
            x = block(x)
        return self.output_layer(x)


# Salary scale constants (log-space normalization, tuned for salary range $30K-$300K)
SALARY_LOG_MEAN = 11.6  # ln(~110K)
SALARY_LOG_STD = 0.45


class SalaryNeuralNetworkInference:
    """Inference wrapper for the salary DNN — mirrors DeepNeuralNetworkInference from week8."""

    def __init__(self):
        self.vectorizer = None
        self.model = None
        self.device = None
        np.random.seed(42)
        torch.manual_seed(42)

    def setup(self):
        self.vectorizer = HashingVectorizer(n_features=5000, stop_words="english", binary=True)
        self.model = SalaryDNN(5000)
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            self.device = torch.device("mps")
        else:
            self.device = torch.device("cpu")
        self.model.to(self.device)
        # Initialize with random weights (in production, load trained weights)
        print(f"Salary DNN initialized on {self.device}")

    def inference(self, text: str) -> float:
        """Predict salary from job description text."""
        self.model.eval()
        with torch.no_grad():
            vector = self.vectorizer.transform([text])
            vector = torch.FloatTensor(vector.toarray()).to(self.device)
            pred = self.model(vector)[0]
            # Convert from log-space prediction to dollar amount
            result = torch.exp(pred * SALARY_LOG_STD + SALARY_LOG_MEAN)
            result = result.item()
        return max(30000, min(result, 500000))  # Clamp to reasonable range


# Test the DNN
salary_dnn = SalaryNeuralNetworkInference()
salary_dnn.setup()
test_salary = salary_dnn.inference("Senior Python Engineer building ML pipelines with PyTorch and AWS")
print(f"DNN salary estimate: ${test_salary:,.0f}")

### MarketAnalysisAgent (Salary Ensemble)
Combines frontier model + DNN in a weighted ensemble — mirrors `ensemble_agent.py` from week8.

In [ ]:
# === MarketAnalysisAgent — Weighted ensemble of frontier + DNN ===

class MarketAnalysisAgent(Agent):
    """Estimates salaries using an ensemble of a frontier model and a DNN — mirrors EnsembleAgent."""

    name = "Market Analysis Agent"
    color = Agent.YELLOW
    FRONTIER_MODEL = "gpt-4o-mini"

    # Ensemble weights
    FRONTIER_WEIGHT = 0.7
    DNN_WEIGHT = 0.3

    def __init__(self):
        self.log("Initializing Market Analysis Agent")
        self.openai = _openai_client
        self.neural_network = SalaryNeuralNetworkInference()
        self.neural_network.setup()
        self.log("Market Analysis Agent is ready (frontier + DNN ensemble)")

    def _frontier_estimate(self, job: Job) -> float:
        """Get salary estimate from the frontier model."""
        self.log(f"Frontier model estimating salary for '{job.title}'")
        prompt = f"""Estimate the annual salary in USD for this job. Respond with ONLY a number (no currency symbol, no commas).

Job Title: {job.title}
Company: {job.company}
Location: {job.location}
Description: {job.description}
Skills: {', '.join(job.tags)}
{f'Salary hint from posting: {job.salary_text}' if job.salary_text else ''}"""

        try:
            response = self.openai.chat.completions.create(
                model=self.FRONTIER_MODEL,
                messages=[
                    {"role": "system", "content": "You are a compensation analyst. Estimate realistic annual salaries based on job details. Respond with only a number."},
                    {"role": "user", "content": prompt},
                ],
            )
            text = response.choices[0].message.content.strip()
            salary = float(re.sub(r"[^\d.]", "", text))
            self.log(f"Frontier estimate: ${salary:,.0f}")
            return salary
        except Exception as e:
            self.log(f"Frontier estimate error: {e}")
            return 100000.0  # fallback

    def _dnn_estimate(self, job: Job) -> float:
        """Get salary estimate from the DNN."""
        self.log(f"DNN estimating salary for '{job.title}'")
        text = f"{job.title} {job.description} {' '.join(job.tags)} {job.location}"
        result = self.neural_network.inference(text)
        self.log(f"DNN estimate: ${result:,.0f}")
        return result

    def estimate_salary(self, job: Job) -> SalaryEstimate:
        """Run the ensemble to estimate salary (frontier + DNN in parallel)."""
        self.log(f"Running ensemble salary estimation for '{job.title}'")

        with ThreadPoolExecutor(max_workers=2) as executor:
            frontier_future = executor.submit(self._frontier_estimate, job)
            dnn_future = executor.submit(self._dnn_estimate, job)
            frontier = frontier_future.result()
            dnn = dnn_future.result()

        combined = frontier * self.FRONTIER_WEIGHT + dnn * self.DNN_WEIGHT
        self.log(f"Ensemble result: ${combined:,.0f} (frontier={self.FRONTIER_WEIGHT}×${frontier:,.0f} + dnn={self.DNN_WEIGHT}×${dnn:,.0f})")

        # Confidence based on how close the two models agree
        diff_pct = abs(frontier - dnn) / max(frontier, dnn, 1)
        if diff_pct < 0.15:
            confidence = "high"
        elif diff_pct < 0.35:
            confidence = "medium"
        else:
            confidence = "low"

        return SalaryEstimate(
            job=job,
            estimated_salary=combined,
            confidence=confidence,
            frontier_estimate=frontier,
            dnn_estimate=dnn,
        )


# Test MarketAnalysisAgent
market_agent = MarketAnalysisAgent()
test_salary_est = market_agent.estimate_salary(test_jobs.jobs[0])
print(f"\nSalary estimate for '{test_salary_est.job.title}':")
print(f"  Estimated: ${test_salary_est.estimated_salary:,.0f}")
print(f"  Frontier: ${test_salary_est.frontier_estimate:,.0f}")
print(f"  DNN: ${test_salary_est.dnn_estimate:,.0f}")
print(f"  Confidence: {test_salary_est.confidence}")

## Phase 6: Notifications & Orchestration

### AlertAgent
Sends push notifications via Pushover API — mirrors `messaging_agent.py` from week8.

In [ ]:
# === AlertAgent — Push notifications via Pushover ===

PUSHOVER_URL = "https://api.pushover.net/1/messages.json"


class AlertAgent(Agent):
    """Sends push notifications about job matches via Pushover — mirrors MessagingAgent."""

    name = "Alert Agent"
    color = Agent.WHITE
    MODEL = "claude-sonnet-4-5"

    def __init__(self):
        self.log("Alert Agent is initializing")
        self.pushover_user = os.getenv("PUSHOVER_USER", "")
        self.pushover_token = os.getenv("PUSHOVER_TOKEN", "")
        self.log("Alert Agent is ready" + (" (Pushover configured)" if self.pushover_user else " (Pushover NOT configured — will log only)"))

    def _push(self, text: str):
        """Send a push notification via Pushover API."""
        if not self.pushover_user or not self.pushover_token:
            self.log(f"[DRY RUN] Push notification: {text[:100]}...")
            return
        self.log("Sending push notification")
        payload = {
            "user": self.pushover_user,
            "token": self.pushover_token,
            "message": text,
            "sound": "magic",
        }
        requests.post(PUSHOVER_URL, data=payload)

    def craft_message(self, match_result: MatchResult, salary_estimate: Optional[SalaryEstimate] = None) -> str:
        """Use Claude via LiteLLM to craft an exciting notification message."""
        job = match_result.job
        salary_info = f"\nEstimated Salary: ${salary_estimate.estimated_salary:,.0f}" if salary_estimate else ""

        prompt = f"""Summarize this job match in 2-3 exciting sentences for a push notification.

Job: {job.title} at {job.company} ({job.location})
Match Score: {match_result.match_score}/100
Matched Skills: {', '.join(match_result.matched_skills)}{salary_info}

Respond only with the 2-3 sentence notification message."""

        try:
            response = completion(
                model=self.MODEL,
                messages=[{"role": "user", "content": prompt}],
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"New job match: {job.title} at {job.company} — Score: {match_result.match_score}/100"

    def alert(self, match_result: MatchResult, salary_estimate: Optional[SalaryEstimate] = None):
        """Send an alert about a great job match."""
        self.log(f"Crafting alert for '{match_result.job.title}'")
        text = self.craft_message(match_result, salary_estimate)
        url = match_result.job.url
        self._push(text[:200] + "... " + url)
        self.log("Alert sent successfully")

    def notify(self, description: str, match_score: float, url: str):
        """Simple notification with description."""
        text = f"Job Match Alert! Score: {match_score}/100 — {description[:150]}... {url}"
        self._push(text)


# Test AlertAgent
alert_agent = AlertAgent()
alert_agent.alert(test_match, test_salary_est)
print("AlertAgent test complete.")

### JobPlanningAgent
Sequential pipeline that orchestrates all agents in order — mirrors `planning_agent.py` from week8.

In [ ]:
# === JobPlanningAgent — Sequential pipeline orchestration ===

class JobPlanningAgent(Agent):
    """Orchestrates the full job hunting pipeline sequentially — mirrors PlanningAgent."""

    name = "Job Planning Agent"
    color = Agent.GREEN
    MATCH_THRESHOLD = 60  # Minimum match score to proceed

    def __init__(self, resume_collection: chromadb.Collection, resume_text: str):
        self.log("Job Planning Agent is initializing")
        self.scanner = JobScannerAgent()
        self.resume_agent = ResumeAgent(resume_collection)
        self.market_agent = MarketAnalysisAgent()
        self.skill_gap_agent = SkillGapAgent()
        self.cover_letter_agent = CoverLetterAgent()
        self.alert_agent = AlertAgent()
        self.resume_text = resume_text
        self.log("Job Planning Agent is ready")

    def process_job(self, job: Job) -> Dict:
        """Run the full pipeline for a single job."""
        self.log(f"Processing job: '{job.title}' at {job.company}")

        # Step 1: Match against resume (RAG)
        match = self.resume_agent.match_job(job)
        result = {"match": match}

        # Step 2: Estimate salary (Ensemble)
        salary = self.market_agent.estimate_salary(job)
        result["salary"] = salary

        # Step 3: Analyze skill gaps (Claude via LiteLLM)
        if match.missing_skills:
            gap = self.skill_gap_agent.analyze(match, self.resume_text)
            result["skill_gap"] = gap

        # Step 4: Generate cover letter for good matches (GPT)
        if match.match_score >= self.MATCH_THRESHOLD:
            letter = self.cover_letter_agent.generate(match, self.resume_text)
            result["cover_letter"] = letter

        # Step 5: Alert on great matches (Pushover)
        if match.match_score >= 75:
            self.alert_agent.alert(match, salary)

        return result

    def plan(self, memory_urls: Optional[List[str]] = None, use_test_data: bool = False) -> Optional[HuntResult]:
        """Run the full job hunting pipeline."""
        if memory_urls is None:
            memory_urls = []
        self.log("Job Planning Agent is kicking off a run")

        # Scan for jobs
        if use_test_data:
            selection = self.scanner.test_scan()
        else:
            selection = self.scanner.scan(memory_urls=memory_urls)

        if not selection or not selection.jobs:
            self.log("No jobs found to process")
            return None

        self.log(f"Processing {len(selection.jobs)} jobs through the pipeline")

        hunt_result = HuntResult()
        for job in selection.jobs:
            result = self.process_job(job)

            hunt_result.matches.append(result["match"])
            hunt_result.salary_estimates.append(result["salary"])

            if "skill_gap" in result:
                hunt_result.skill_gaps.append(result["skill_gap"])
            if "cover_letter" in result:
                hunt_result.cover_letters.append(result["cover_letter"])

        # Sort by match score and pick the best
        hunt_result.matches.sort(key=lambda m: m.match_score, reverse=True)
        if hunt_result.matches:
            hunt_result.best_match = hunt_result.matches[0]

        self.log(f"Pipeline complete: {len(hunt_result.matches)} matches, "
                 f"{len(hunt_result.cover_letters)} cover letters, "
                 f"best score: {hunt_result.best_match.match_score if hunt_result.best_match else 'N/A'}")
        return hunt_result


print("JobPlanningAgent defined.")

### AutonomousHuntingAgent
LLM-driven tool-use agentic loop — the LLM decides which tools to call and in what order. Mirrors `autonomous_planning_agent.py` from week8 (Week 3-4, 8 skills).

In [ ]:
# === AutonomousHuntingAgent — LLM tool-use agentic loop ===

class AutonomousHuntingAgent(Agent):
    """
    LLM-driven autonomous agent that uses tool calling to orchestrate the job hunting pipeline.
    The LLM decides which tools to call and in what order — mirrors AutonomousPlanningAgent.
    """

    name = "Autonomous Hunting Agent"
    color = Agent.GREEN
    MODEL = "gpt-4o-mini"

    def __init__(self, resume_collection: chromadb.Collection, resume_text: str):
        self.log("Autonomous Hunting Agent is initializing")
        self.scanner = JobScannerAgent()
        self.resume_agent = ResumeAgent(resume_collection)
        self.market_agent = MarketAnalysisAgent()
        self.skill_gap_agent = SkillGapAgent()
        self.cover_letter_agent = CoverLetterAgent()
        self.alert_agent = AlertAgent()
        self.resume_text = resume_text
        self.openai = _openai_client
        self.memory_urls = []
        self.hunt_result = HuntResult()
        self.log("Autonomous Hunting Agent is ready")

    # --- Shared helper ---

    @staticmethod
    def _build_job(job_title: str, job_company: str = "", job_description: str = "",
                   job_location: str = "", job_url: str = "", job_tags: str = "") -> Job:
        """Build a Job from tool-call string arguments."""
        return Job(
            title=job_title,
            company=job_company,
            description=job_description,
            location=job_location,
            url=job_url,
            tags=[t.strip() for t in job_tags.split(",")] if job_tags else [],
        )

    # --- Tool implementations ---

    def scan_for_jobs(self) -> str:
        self.log("Tool called: scan_for_jobs")
        results = self.scanner.test_scan()  # Use test data for demo; swap to self.scanner.scan() for live
        return results.model_dump_json() if results else "No jobs found"

    def match_job_to_resume(self, job_title: str, job_company: str, job_description: str,
                             job_location: str, job_url: str, job_tags: str) -> str:
        self.log(f"Tool called: match_job_to_resume for '{job_title}'")
        job = self._build_job(job_title, job_company, job_description, job_location, job_url, job_tags)
        match = self.resume_agent.match_job(job)
        self.hunt_result.matches.append(match)
        return match.model_dump_json()

    def estimate_salary(self, job_title: str, job_company: str, job_description: str,
                         job_location: str, job_tags: str) -> str:
        self.log(f"Tool called: estimate_salary for '{job_title}'")
        job = self._build_job(job_title, job_company, job_description, job_location, job_tags=job_tags)
        estimate = self.market_agent.estimate_salary(job)
        self.hunt_result.salary_estimates.append(estimate)
        return f"Estimated salary for {job_title}: ${estimate.estimated_salary:,.0f} (confidence: {estimate.confidence})"

    def analyze_skill_gaps(self, job_title: str, missing_skills: str) -> str:
        self.log(f"Tool called: analyze_skill_gaps for '{job_title}'")
        job = self._build_job(job_title)
        match = MatchResult(job=job, match_score=0, missing_skills=[s.strip() for s in missing_skills.split(",")])
        gap = self.skill_gap_agent.analyze(match, self.resume_text)
        self.hunt_result.skill_gaps.append(gap)
        return json.dumps({"missing_skills": gap.missing_skills, "priority": gap.priority,
                           "estimated_weeks": gap.estimated_weeks,
                           "resources": gap.learning_resources[:3]})

    def generate_cover_letter(self, job_title: str, job_company: str, job_description: str,
                               matched_skills: str) -> str:
        self.log(f"Tool called: generate_cover_letter for '{job_title}'")
        job = self._build_job(job_title, job_company, job_description)
        match = MatchResult(job=job, match_score=80,
                            matched_skills=[s.strip() for s in matched_skills.split(",")])
        letter = self.cover_letter_agent.generate(match, self.resume_text)
        self.hunt_result.cover_letters.append(letter)
        return f"Cover letter generated ({len(letter.letter_text)} chars). Key points: {letter.key_points}"

    def notify_user(self, message: str, url: str) -> str:
        self.log(f"Tool called: notify_user")
        self.alert_agent._push(message[:200] + "... " + url)
        return "Notification sent"

    # --- Tool definitions for OpenAI function calling ---

    TOOLS = [
        {
            "type": "function",
            "function": {
                "name": "scan_for_jobs",
                "description": "Scan the internet for remote tech job postings from RSS feeds",
                "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": False},
            },
        },
        {
            "type": "function",
            "function": {
                "name": "match_job_to_resume",
                "description": "Match a specific job against the candidate's resume using RAG and return a match score",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "job_title": {"type": "string", "description": "The job title"},
                        "job_company": {"type": "string", "description": "The company name"},
                        "job_description": {"type": "string", "description": "The job description"},
                        "job_location": {"type": "string", "description": "The job location"},
                        "job_url": {"type": "string", "description": "URL of the job posting"},
                        "job_tags": {"type": "string", "description": "Comma-separated skill tags"},
                    },
                    "required": ["job_title", "job_company", "job_description", "job_location", "job_url", "job_tags"],
                    "additionalProperties": False,
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "estimate_salary",
                "description": "Estimate the salary for a job using an ensemble of frontier model and neural network",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "job_title": {"type": "string"},
                        "job_company": {"type": "string"},
                        "job_description": {"type": "string"},
                        "job_location": {"type": "string"},
                        "job_tags": {"type": "string", "description": "Comma-separated skill tags"},
                    },
                    "required": ["job_title", "job_company", "job_description", "job_location", "job_tags"],
                    "additionalProperties": False,
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "analyze_skill_gaps",
                "description": "Analyze what skills the candidate is missing for a job and recommend learning resources",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "job_title": {"type": "string"},
                        "missing_skills": {"type": "string", "description": "Comma-separated missing skills"},
                    },
                    "required": ["job_title", "missing_skills"],
                    "additionalProperties": False,
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "generate_cover_letter",
                "description": "Generate a tailored cover letter for a job the candidate is a good match for",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "job_title": {"type": "string"},
                        "job_company": {"type": "string"},
                        "job_description": {"type": "string"},
                        "matched_skills": {"type": "string", "description": "Comma-separated matched skills"},
                    },
                    "required": ["job_title", "job_company", "job_description", "matched_skills"],
                    "additionalProperties": False,
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "notify_user",
                "description": "Send the user a push notification about a great job match; only call once for the best match",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "message": {"type": "string", "description": "The notification message"},
                        "url": {"type": "string", "description": "URL of the job posting"},
                    },
                    "required": ["message", "url"],
                    "additionalProperties": False,
                },
            },
        },
    ]

    def handle_tool_call(self, message) -> List[Dict]:
        """Dispatch tool calls from the LLM response."""
        tool_map = {
            "scan_for_jobs": self.scan_for_jobs,
            "match_job_to_resume": self.match_job_to_resume,
            "estimate_salary": self.estimate_salary,
            "analyze_skill_gaps": self.analyze_skill_gaps,
            "generate_cover_letter": self.generate_cover_letter,
            "notify_user": self.notify_user,
        }
        results = []
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            tool = tool_map.get(tool_name)
            result = tool(**arguments) if tool else f"Unknown tool: {tool_name}"
            results.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
        return results

    SYSTEM_MESSAGE = """You are an AI Job Hunter agent. You help candidates find the best job matches using your tools.

Your workflow:
1. First, scan for available jobs
2. For each promising job, match it against the candidate's resume
3. For the best matches (score > 60), estimate the salary and analyze skill gaps
4. For the top match, generate a cover letter
5. Notify the user about the single best opportunity

Be efficient — focus on the most promising jobs first."""

    USER_MESSAGE = """Please hunt for jobs for me. Scan for available positions, match them against my resume,
estimate salaries for the best matches, identify any skill gaps, generate a cover letter for the top match,
and notify me about the best opportunity. Then reply with a summary of what you found."""

    def hunt(self) -> HuntResult:
        """Run the autonomous hunting loop — LLM drives tool selection."""
        self.log("Autonomous Hunting Agent is starting a hunt")
        self.hunt_result = HuntResult()

        messages = [
            {"role": "system", "content": self.SYSTEM_MESSAGE},
            {"role": "user", "content": self.USER_MESSAGE},
        ]

        done = False
        iterations = 0
        max_iterations = 15  # Safety limit

        while not done and iterations < max_iterations:
            iterations += 1
            self.log(f"LLM iteration {iterations}")

            response = self.openai.chat.completions.create(
                model=self.MODEL,
                messages=messages,
                tools=self.TOOLS,
            )

            if response.choices[0].finish_reason == "tool_calls":
                message = response.choices[0].message
                tool_results = self.handle_tool_call(message)
                messages.append(message)
                messages.extend(tool_results)
            else:
                done = True

        reply = response.choices[0].message.content
        self.log(f"Autonomous hunt completed after {iterations} iterations")
        self.log(f"Summary: {reply[:200] if reply else 'No summary'}")

        # Set best match
        if self.hunt_result.matches:
            self.hunt_result.matches.sort(key=lambda m: m.match_score, reverse=True)
            self.hunt_result.best_match = self.hunt_result.matches[0]

        return self.hunt_result


print("AutonomousHuntingAgent defined with 6 tools.")

### JobHunterFramework
Central coordinator with ChromaDB + memory.json — mirrors `deal_agent_framework.py` from week8.

In [ ]:
# === JobHunterFramework — Central coordinator with memory ===

class JobHunterFramework:
    """
    Central coordinator for the AI Job Hunter system.
    Manages ChromaDB vectorstore, persistent memory, and agent orchestration.
    Mirrors DealAgentFramework from week8.
    """

    MEMORY_FILENAME = "job_hunter_memory.json"
    MAX_MEMORY_ENTRIES = 200

    def __init__(self, resume_text: str = SAMPLE_RESUME, use_autonomous: bool = False):
        init_logging()
        self.log("Initializing Job Hunter Framework")
        self.resume_text = resume_text

        # Build resume vectorstore
        self.resume_collection = build_resume_vectorstore(resume_text)
        self.memory = self.read_memory()
        self.use_autonomous = use_autonomous
        self.planner = None
        self.autonomous = None
        self.last_result = None
        self.log("Job Hunter Framework initialized")

    def init_agents_as_needed(self):
        if self.use_autonomous and not self.autonomous:
            self.log("Initializing Autonomous Hunting Agent")
            self.autonomous = AutonomousHuntingAgent(self.resume_collection, self.resume_text)
        elif not self.planner:
            self.log("Initializing Job Planning Agent")
            self.planner = JobPlanningAgent(self.resume_collection, self.resume_text)
        self.log("Agents ready")

    def read_memory(self) -> List[Dict]:
        if os.path.exists(self.MEMORY_FILENAME):
            with open(self.MEMORY_FILENAME, "r") as f:
                return json.load(f)
        return []

    def write_memory(self):
        # Deduplicate by URL, keeping the most recent entry
        seen_urls = {}
        for entry in self.memory:
            url = entry.get("url", "")
            seen_urls[url] = entry  # later entries overwrite earlier ones
        self.memory = list(seen_urls.values())

        # Prune to last MAX_MEMORY_ENTRIES
        if len(self.memory) > self.MAX_MEMORY_ENTRIES:
            self.memory = self.memory[-self.MAX_MEMORY_ENTRIES:]

        with open(self.MEMORY_FILENAME, "w") as f:
            json.dump(self.memory, f, indent=2)

    @classmethod
    def reset_memory(cls):
        if os.path.exists(cls.MEMORY_FILENAME):
            with open(cls.MEMORY_FILENAME, "w") as f:
                json.dump([], f)

    def log(self, message: str):
        text = BG_BLUE + Agent.WHITE + "[Job Hunter Framework] " + message + Agent.RESET
        logging.info(text)

    def run(self, use_test_data: bool = True) -> Optional[HuntResult]:
        """Run the job hunting pipeline."""
        self.init_agents_as_needed()
        self.log("Starting job hunt run")

        memory_urls = [m.get("url", "") for m in self.memory]

        if self.use_autonomous:
            result = self.autonomous.hunt()
        else:
            result = self.planner.plan(memory_urls=memory_urls, use_test_data=use_test_data)

        if result and result.matches:
            # Save matches to memory
            for match in result.matches:
                entry = {
                    "url": match.job.url,
                    "title": match.job.title,
                    "company": match.job.company,
                    "match_score": match.match_score,
                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                }
                self.memory.append(entry)
            self.write_memory()

        self.last_result = result
        self.log(f"Run complete: {len(result.matches) if result else 0} matches found")
        return result

    def get_plot_data(self):
        """Get resume vectorstore data for 3D visualization."""
        result = self.resume_collection.get(include=["embeddings", "documents"])
        vectors = np.array(result["embeddings"])
        documents = result["documents"]
        if len(vectors) < 4:
            return documents, vectors, ["blue"] * len(vectors)
        tsne = TSNE(n_components=3, random_state=42, perplexity=min(3, len(vectors) - 1))
        reduced = tsne.fit_transform(vectors)
        colors = ["blue"] * len(vectors)
        return documents, reduced, colors


print("JobHunterFramework defined.")

## Phase 7: Gradio UI

5-tab dashboard with:
- **Job Matches** — dataframe + live agent logs + match score chart
- **Skill Gaps** — radar chart + learning resources table
- **Salary Insights** — range plots + ensemble breakdown
- **Cover Letters** — job selector dropdown + generated text
- **Resume** — paste/ingest + 3D t-SNE vector visualization

Mirrors `price_is_right.py` from week8, including QueueHandler logging and Timer.

In [ ]:
# === Gradio UI — 5-tab Job Hunter Dashboard ===

class QueueHandler(logging.Handler):
    """Custom logging handler that puts messages into a queue for Gradio display."""
    def __init__(self, log_queue):
        super().__init__()
        self.log_queue = log_queue

    def emit(self, record):
        self.log_queue.put(self.format(record))


def html_for(log_data):
    output = "<br>".join(log_data[-20:])
    return f"""
    <div style="height: 400px; overflow-y: auto; border: 1px solid #ccc; background-color: #222229; padding: 10px; font-family: monospace; font-size: 12px;">
    {output}
    </div>
    """


def setup_ui_logging(log_queue):
    handler = QueueHandler(log_queue)
    formatter = logging.Formatter("[%(asctime)s] %(message)s", datefmt="%H:%M:%S")
    handler.setFormatter(formatter)
    logger = logging.getLogger()
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)


class JobHunterApp:
    def __init__(self):
        self.framework = None
        self.last_result = None

    def get_framework(self):
        if not self.framework:
            self.framework = JobHunterFramework()
        return self.framework

    def run(self):
        with gr.Blocks(title="AI Job Hunter", fill_width=True, theme=gr.themes.Soft()) as ui:
            log_data = gr.State([])

            # --- Header ---
            gr.Markdown(
                '<div style="text-align: center; font-size: 28px; font-weight: bold;">AI Job Hunter</div>'
                '<div style="text-align: center; font-size: 14px; color: #666;">Autonomous Multi-Agent System — Capstone Project</div>'
            )

            with gr.Tabs():
                # === Tab 1: Job Matches ===
                with gr.Tab("Job Matches"):
                    with gr.Row():
                        run_btn = gr.Button("Hunt for Jobs (Test Data)", variant="primary", scale=1)
                        run_live_btn = gr.Button("Hunt for Jobs (Live RSS)", variant="secondary", scale=1)
                        autonomous_btn = gr.Button("Autonomous Hunt", variant="secondary", scale=1)

                    matches_df = gr.Dataframe(
                        headers=["Job Title", "Company", "Location", "Match Score", "Matched Skills", "Missing Skills"],
                        wrap=True,
                        column_widths=[2, 1.5, 1, 1, 2, 2],
                        row_count=10,
                        max_height=350,
                    )

                    with gr.Row():
                        with gr.Column(scale=1):
                            logs_html = gr.HTML(value="<div style='padding:10px; color:#666;'>Logs will appear here when you start a hunt...</div>")
                        with gr.Column(scale=1):
                            match_chart = gr.Plot(label="Match Scores")

                # === Tab 2: Skill Gaps ===
                with gr.Tab("Skill Gaps"):
                    skill_gaps_df = gr.Dataframe(
                        headers=["Job Title", "Missing Skills", "Priority", "Est. Weeks", "Top Resource"],
                        wrap=True,
                        row_count=10,
                        max_height=300,
                    )
                    with gr.Row():
                        with gr.Column():
                            radar_chart = gr.Plot(label="Skills Radar")
                        with gr.Column():
                            resources_df = gr.Dataframe(
                                headers=["Skill", "Resource", "Type"],
                                wrap=True,
                                row_count=10,
                                max_height=300,
                            )

                # === Tab 3: Salary Insights ===
                with gr.Tab("Salary Insights"):
                    salary_df = gr.Dataframe(
                        headers=["Job Title", "Company", "Estimated Salary", "Frontier Est.", "DNN Est.", "Confidence"],
                        wrap=True,
                        row_count=10,
                        max_height=300,
                    )
                    salary_chart = gr.Plot(label="Salary Estimates")

                # === Tab 4: Cover Letters ===
                with gr.Tab("Cover Letters"):
                    with gr.Row():
                        letter_selector = gr.Dropdown(label="Select Job", choices=[], interactive=True)
                    letter_output = gr.Textbox(label="Generated Cover Letter", lines=20, interactive=False)
                    key_points_output = gr.Textbox(label="Key Selling Points", lines=5, interactive=False)

                # === Tab 5: Resume ===
                with gr.Tab("Resume"):
                    with gr.Row():
                        with gr.Column(scale=1):
                            resume_input = gr.Textbox(
                                label="Resume Text",
                                value=SAMPLE_RESUME,
                                lines=20,
                                interactive=True,
                            )
                            ingest_btn = gr.Button("Re-ingest Resume", variant="secondary")
                        with gr.Column(scale=1):
                            vector_plot = gr.Plot(label="Resume Chunk Embeddings (3D t-SNE)")

            # --- Helper functions ---

            def make_match_table(result):
                if not result or not result.matches:
                    return []
                return [
                    [m.job.title, m.job.company, m.job.location,
                     f"{m.match_score}/100", ", ".join(m.matched_skills[:4]),
                     ", ".join(m.missing_skills[:4])]
                    for m in result.matches
                ]

            def make_match_chart(result):
                if not result or not result.matches:
                    return go.Figure()
                titles = [m.job.title[:25] for m in result.matches]
                scores = [m.match_score for m in result.matches]
                colors = ["#2ecc71" if s >= 75 else "#f39c12" if s >= 50 else "#e74c3c" for s in scores]
                fig = go.Figure(go.Bar(x=titles, y=scores, marker_color=colors))
                fig.update_layout(title="Match Scores", yaxis_title="Score", height=350,
                                  yaxis=dict(range=[0, 100]))
                return fig

            def make_skill_gaps_table(result):
                if not result or not result.skill_gaps:
                    return []
                return [
                    [g.job.title, ", ".join(g.missing_skills[:4]), g.priority,
                     str(g.estimated_weeks),
                     g.learning_resources[0].get("resource", "N/A") if g.learning_resources else "N/A"]
                    for g in result.skill_gaps
                ]

            def make_radar_chart(result):
                if not result or not result.matches:
                    return go.Figure()
                # Aggregate all skills across matches
                all_skills = {}
                for m in result.matches:
                    for s in m.matched_skills:
                        all_skills[s] = all_skills.get(s, 0) + 1
                    for s in m.missing_skills:
                        if s not in all_skills:
                            all_skills[s] = 0
                if not all_skills:
                    return go.Figure()
                top_skills = sorted(all_skills.items(), key=lambda x: x[1], reverse=True)[:10]
                categories = [s[0] for s in top_skills]
                values = [s[1] for s in top_skills]
                fig = go.Figure(go.Scatterpolar(r=values + [values[0]],
                                                theta=categories + [categories[0]],
                                                fill="toself", name="Skill Coverage"))
                fig.update_layout(title="Skills Radar", height=350,
                                  polar=dict(radialaxis=dict(visible=True)))
                return fig

            def make_resources_table(result):
                if not result or not result.skill_gaps:
                    return []
                rows = []
                for gap in result.skill_gaps:
                    for r in gap.learning_resources[:3]:
                        rows.append([r.get("skill", ""), r.get("resource", ""), r.get("type", "")])
                return rows

            def make_salary_table(result):
                if not result or not result.salary_estimates:
                    return []
                return [
                    [s.job.title, s.job.company,
                     f"${s.estimated_salary:,.0f}",
                     f"${s.frontier_estimate:,.0f}",
                     f"${s.dnn_estimate:,.0f}",
                     s.confidence]
                    for s in result.salary_estimates
                ]

            def make_salary_chart(result):
                if not result or not result.salary_estimates:
                    return go.Figure()
                titles = [s.job.title[:25] for s in result.salary_estimates]
                ensemble = [s.estimated_salary for s in result.salary_estimates]
                frontier = [s.frontier_estimate for s in result.salary_estimates]
                dnn = [s.dnn_estimate for s in result.salary_estimates]
                fig = go.Figure()
                fig.add_trace(go.Bar(name="Ensemble", x=titles, y=ensemble, marker_color="#3498db"))
                fig.add_trace(go.Bar(name="Frontier", x=titles, y=frontier, marker_color="#2ecc71"))
                fig.add_trace(go.Bar(name="DNN", x=titles, y=dnn, marker_color="#e74c3c"))
                fig.update_layout(title="Salary Estimates Breakdown", barmode="group",
                                  yaxis_title="Annual Salary ($)", height=350)
                return fig

            def get_letter_choices(result):
                if not result or not result.cover_letters:
                    return gr.update(choices=[], value=None)
                choices = [f"{l.job.title} @ {l.job.company}" for l in result.cover_letters]
                return gr.update(choices=choices, value=choices[0] if choices else None)

            def show_letter(selection, result):
                if not result or not result.cover_letters or not selection:
                    return "", ""
                for letter in result.cover_letters:
                    if f"{letter.job.title} @ {letter.job.company}" == selection:
                        return letter.letter_text, "\n".join([f"• {p}" for p in letter.key_points])
                return "", ""

            def make_vector_plot():
                try:
                    fw = self.get_framework()
                    documents, vectors, colors = fw.get_plot_data()
                    if vectors.shape[1] < 3:
                        return go.Figure().update_layout(title="Need more chunks for 3D visualization")
                    fig = go.Figure(data=[go.Scatter3d(
                        x=vectors[:, 0], y=vectors[:, 1], z=vectors[:, 2],
                        mode="markers",
                        marker=dict(size=6, color=colors, opacity=0.8),
                        text=[d[:80] for d in documents],
                    )])
                    fig.update_layout(title="Resume Chunk Embeddings", height=400,
                                      scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"))
                    return fig
                except Exception:
                    return go.Figure()

            # --- Event handlers ---

            def do_hunt(initial_log_data, use_test: bool = True, autonomous: bool = False):
                log_queue = queue.Queue()
                result_queue = queue.Queue()
                setup_ui_logging(log_queue)

                def worker():
                    fw = self.get_framework()
                    if autonomous:
                        fw.use_autonomous = True
                        fw.autonomous = None  # Force re-init
                    else:
                        fw.use_autonomous = False
                    r = fw.run(use_test_data=use_test)
                    result_queue.put(r)

                thread = threading.Thread(target=worker)
                thread.start()

                result = None
                while True:
                    try:
                        msg = log_queue.get_nowait()
                        initial_log_data.append(reformat(msg))
                        yield (initial_log_data, html_for(initial_log_data),
                               make_match_table(result), make_match_chart(result),
                               make_skill_gaps_table(result), make_radar_chart(result),
                               make_resources_table(result),
                               make_salary_table(result), make_salary_chart(result),
                               get_letter_choices(result))
                    except queue.Empty:
                        try:
                            result = result_queue.get_nowait()
                            self.last_result = result
                            yield (initial_log_data, html_for(initial_log_data),
                                   make_match_table(result), make_match_chart(result),
                                   make_skill_gaps_table(result), make_radar_chart(result),
                                   make_resources_table(result),
                                   make_salary_table(result), make_salary_chart(result),
                                   get_letter_choices(result))
                        except queue.Empty:
                            if result is not None:
                                break
                            time.sleep(0.1)

            outputs = [log_data, logs_html, matches_df, match_chart,
                       skill_gaps_df, radar_chart, resources_df,
                       salary_df, salary_chart, letter_selector]

            run_btn.click(
                lambda ld: do_hunt(ld, use_test=True, autonomous=False),
                inputs=[log_data], outputs=outputs,
            )
            run_live_btn.click(
                lambda ld: do_hunt(ld, use_test=False, autonomous=False),
                inputs=[log_data], outputs=outputs,
            )
            autonomous_btn.click(
                lambda ld: do_hunt(ld, use_test=True, autonomous=True),
                inputs=[log_data], outputs=outputs,
            )

            letter_selector.change(
                lambda sel: show_letter(sel, self.last_result),
                inputs=[letter_selector],
                outputs=[letter_output, key_points_output],
            )

            def reingest_resume(text):
                self.framework = None  # Force re-init with new resume
                global SAMPLE_RESUME
                SAMPLE_RESUME = text
                return make_vector_plot()

            ingest_btn.click(reingest_resume, inputs=[resume_input], outputs=[vector_plot])

            # Load vector plot on startup
            ui.load(make_vector_plot, outputs=[vector_plot])

        ui.launch(share=False, inbrowser=True)


print("JobHunterApp Gradio UI defined with 5 tabs.")

## Run the Pipeline

### Option A: Run the sequential pipeline directly (test data)

In [ ]:
# Run the full sequential pipeline with test data
planner = JobPlanningAgent(resume_collection, SAMPLE_RESUME)
result = planner.plan(use_test_data=True)

if result:
    print(f"\n{'='*60}")
    print(f"HUNT RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"Total matches: {len(result.matches)}")
    print(f"Cover letters generated: {len(result.cover_letters)}")
    print(f"Salary estimates: {len(result.salary_estimates)}")
    print(f"Skill gap analyses: {len(result.skill_gaps)}")
    if result.best_match:
        print(f"\nBest match: {result.best_match.job.title} @ {result.best_match.job.company}")
        print(f"  Score: {result.best_match.match_score}/100")
        print(f"  Matched: {result.best_match.matched_skills}")
    print(f"\nAll matches:")
    for m in result.matches:
        print(f"  [{m.match_score:3d}/100] {m.job.title} @ {m.job.company}")
else:
    print("No results returned.")

### Option B: Run the Autonomous Hunting Agent (LLM-driven tool use)

In [ ]:
# Run the autonomous agent — LLM decides which tools to call
autonomous = AutonomousHuntingAgent(resume_collection, SAMPLE_RESUME)
auto_result = autonomous.hunt()

if auto_result:
    print(f"\n{'='*60}")
    print(f"AUTONOMOUS HUNT RESULTS")
    print(f"{'='*60}")
    print(f"Matches found: {len(auto_result.matches)}")
    print(f"Salary estimates: {len(auto_result.salary_estimates)}")
    print(f"Skill gaps analyzed: {len(auto_result.skill_gaps)}")
    print(f"Cover letters: {len(auto_result.cover_letters)}")
    if auto_result.best_match:
        print(f"\nBest: {auto_result.best_match.job.title} — {auto_result.best_match.match_score}/100")

### Option C: Launch the Gradio Dashboard
Run the cell below to launch the full 5-tab Gradio UI.

In [ ]:
# Launch the Gradio dashboard
app = JobHunterApp()
app.run()

## Skills Demonstrated

| Week | Skill | Component in This Project |
|------|-------|---------------------------|
| 1-2 | LLM APIs, Prompt Engineering | CoverLetterAgent, SkillGapAgent, all system prompts |
| 3-4 | Advanced LLM Apps (Function Calling) | AutonomousHuntingAgent (6-tool agentic loop) |
| 5 | RAG, ChromaDB, Gradio | ResumeAgent, build_vectorstore, 5-tab Gradio UI |
| 6 | ML/DL, Data Curation | SalaryDNN (PyTorch with ResidualBlocks) |
| 7 | Fine-tuning, QLoRA | SalaryDNN architecture (mirrors week8 DNN) |
| 8 | Multi-Agent, Ensemble, Tool Use, Notifications | JobHunterFramework, MarketAnalysisAgent ensemble, AlertAgent |

## Verification Checklist
1. `JobScannerAgent.test_scan()` — returns 5 test job postings
2. `build_resume_vectorstore()` — populates ChromaDB with resume chunks
3. `ResumeAgent.match_job()` — returns match scores with reasoning
4. `MarketAnalysisAgent.estimate_salary()` — returns ensemble salary estimate
5. `SkillGapAgent.analyze()` — returns skill gaps + learning resources
6. `CoverLetterAgent.generate()` — returns tailored cover letter
7. `JobPlanningAgent.plan()` — runs full sequential pipeline
8. `AutonomousHuntingAgent.hunt()` — LLM drives tool-use loop
9. `JobHunterApp.run()` — launches 5-tab Gradio dashboard